# 📦 Étape 1 — Exploration des données GTFS SNCB

**Objectif :** Comprendre la structure des données GTFS statiques (horaires, gares, itinéraires)

**Source :** Belgian Mobility Data — https://data.belgianmobility.io

**Format :** GTFS (General Transit Feed Specification) — standard mondial transport public

## ⚙️ Installation & Imports

In [ ]:
import pandas as pd
import requests
import zipfile
import os
import io
import warnings
warnings.filterwarnings('ignore')

# Dossiers
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/clean', exist_ok=True)

print('✅ Imports OK')

## 📥 Téléchargement des données GTFS SNCB

Le GTFS est un **fichier ZIP** contenant plusieurs fichiers CSV :
- `stops.txt` — Toutes les gares/arrêts avec coordonnées GPS
- `routes.txt` — Les lignes (IC, L, S, P...)
- `trips.txt` — Les trajets individuels
- `stop_times.txt` — Les horaires à chaque arrêt
- `calendar.txt` — Les jours de service

In [ ]:
# URL du GTFS SNCB (à vérifier sur Belgian Mobility Data)
GTFS_URL = 'https://gtfs.irail.be/sncb/nmbs-latest.zip'

# Téléchargement
print('📥 Téléchargement GTFS SNCB...')
resp = requests.get(GTFS_URL, timeout=120)
resp.raise_for_status()
print(f'   ✅ {len(resp.content) / 1024 / 1024:.1f} Mo téléchargés')

# Extraction des fichiers CSV
zip_file = zipfile.ZipFile(io.BytesIO(resp.content))
print(f'\n📁 Fichiers dans le ZIP :')
for name in zip_file.namelist():
    info = zip_file.getinfo(name)
    print(f'   - {name} ({info.file_size / 1024:.1f} Ko)')

In [ ]:
# Charger chaque fichier CSV dans un DataFrame
dataframes = {}
for name in zip_file.namelist():
    if name.endswith('.txt'):
        df_name = name.replace('.txt', '')
        dataframes[df_name] = pd.read_csv(zip_file.open(name))
        print(f'   ✅ {df_name} : {len(dataframes[df_name]):,} lignes × {len(dataframes[df_name].columns)} colonnes')

print(f'\n📊 Total : {len(dataframes)} tables chargées')

## 🚉 Exploration — Les gares (stops)

In [ ]:
df_stops = dataframes['stops']
print(f'Nombre de gares/arrêts : {len(df_stops):,}')
print(f'\nColonnes : {list(df_stops.columns)}')
df_stops.head(10)

In [ ]:
# Vérifier les coordonnées GPS
print('Valeurs manquantes :')
print(df_stops.isnull().sum())
print(f'\nGares sans coordonnées : {df_stops["stop_lat"].isna().sum()}')

## 🛤️ Exploration — Les lignes (routes)

In [ ]:
df_routes = dataframes['routes']
print(f'Nombre de lignes : {len(df_routes):,}')
print(f'\nTypes de service :')
if 'route_short_name' in df_routes.columns:
    print(df_routes['route_short_name'].value_counts().head(20))
df_routes.head(10)

## 🚂 Exploration — Les trajets (trips)

In [ ]:
df_trips = dataframes['trips']
print(f'Nombre de trajets : {len(df_trips):,}')
print(f'\nColonnes : {list(df_trips.columns)}')
df_trips.head(10)

## 🔗 Croisement — Lignes × Trajets

In [ ]:
# Nombre de trajets par type de train
if 'route_id' in df_trips.columns and 'route_id' in df_routes.columns:
    merged = df_trips.merge(df_routes[['route_id', 'route_short_name']], on='route_id')
    print('🚆 Trajets par type de train :')
    print(merged['route_short_name'].value_counts().head(10))

## 📝 Conclusions & Prochaine étape

### Ce qu'on a appris :
- Structure des données GTFS SNCB
- Nombre de gares, lignes, trajets
- Disponibilité des coordonnées GPS

### Prochaine étape :
→ **02_api_temps_reel.ipynb** — Connexion à l'API GTFS-Realtime